# Caliber — production monitoring

Your eval looks fine in dev. You ship it. A few weeks later production
starts behaving differently — model upgraded under you, user mix shifted,
a tool dependency changed. You want to catch this without staring at a
dashboard.

Caliber ships two streaming change-point detectors:

- **Page-Hinkley** — learns the baseline from the data; use when you
  don't know the target mean in advance.
- **CUSUM** — tighter and more sensitive when you *do* know the
  target mean and noise scale.

In [ ]:
import numpy as np
import caliber

print('caliber', caliber.__version__)

## Simulate a production score stream

600 samples. The first 300 are at mean 0.70 (stable). At sample 300 the
score drops to 0.55 — something changed.

In [ ]:
rng = np.random.default_rng(0)
stationary = rng.normal(0.70, 0.10, 300)
drifted    = rng.normal(0.55, 0.10, 300)
stream     = np.concatenate([stationary, drifted])
print('stream length:', len(stream))
print(f'mean before idx 300: {stationary.mean():.3f}')
print(f'mean after  idx 300: {drifted.mean():.3f}')

## Page-Hinkley

Default parameters `delta=0.05, threshold=50` are very conservative — good
for low false-alarm rate. For faster detection use a smaller threshold
tuned to your noise scale; for fewer false alarms use a larger one.

In [ ]:
detector = caliber.PageHinkleyDetector(delta=0.05, threshold=5.0)

for i, score in enumerate(stream):
    event = detector.add(float(score))
    if event is not None:
        print(
            f'fired at sample {i}: '
            f'mean {event.mean_before:.3f} -> {event.mean_after:.3f}  '
            f'(magnitude {event.magnitude:.3f}, '
            f'post-hoc p={event.p_value:.3g})'
        )
        break  # one alarm is enough for the demo
else:
    print('no drift detected')

## CUSUM — when the target is known

If you can say "the score should be 0.70 ± 0.10 under H₀", CUSUM is the
stronger detector. Same setup, different math.

Note CUSUM doesn't know the shift is the new normal — if you don't
reset and adopt the new baseline, it will keep alarming. In production
you'd typically alert once, page someone, and decide whether to
re-target.

In [ ]:
detector = caliber.CUSUMDetector(
    target_mean=0.70,
    target_std=0.10,
    k=0.5,  # half the shift size you want to catch quickly
    h=8.0,  # decision threshold in σ units — higher = more conservative
)

for i, score in enumerate(stream):
    event = detector.add(float(score))
    if event is not None:
        print(
            f'fired at sample {i}: '
            f'mean {event.mean_before:.3f} -> {event.mean_after:.3f}  '
            f'(magnitude {event.magnitude:.3f}, '
            f'post-hoc p={event.p_value:.3g})'
        )
        break  # one alarm is enough; CUSUM keeps firing until you retarget
else:
    print('no drift detected')

## Choosing between them

| Detector | When | Trade-off |
|---|---|---|
| `PageHinkleyDetector` | You don't know the target mean | Learns baseline; slower |
| `CUSUMDetector` | You know the target mean + σ | Tighter detection; needs target |

Both have well-studied false-positive properties — start with default
parameters and tune `threshold` / `h` based on observed false-alarm rate.

## Production integration

In a real service this lives in your score-logging path:

```python
detector = caliber.PageHinkleyDetector()  # at process start

def on_eval_score(score: float) -> None:
    event = detector.add(score)
    if event is not None:
        emit_alert(
            'Eval score drift detected',
            mean_before=event.mean_before,
            mean_after=event.mean_after,
            magnitude=event.magnitude,
        )
        detector.reset()
```

Or run it offline over a CSV from the shell:

```bash
caliber drift scores.csv --threshold 5
```